In [1]:
input_bam_file = "/home/wbguo/iproject/BSReadSim/test/data/sim/pe_d/sim.mkdup.sorted.bam"
output_bam_file= "/home/wbguo/iproject/BSReadSim/test/data/sim/pe_d/output_C_C2T.bam"
num_threads = 12

In [2]:
import pysam
import concurrent.futures
from itertools import islice
import threading

def process_read(read):
    # Check if the read has the tag YS:Z:C_C2T
    if 'YS' in read.tags and read.get_tag('YS') == 'C_C2T':
        return read
    else:
        return None

def extract_reads(input_bam_filename, output_bam_filename, num_processes):
    # Open input and output bam files
    with pysam.AlignmentFile(input_bam_filename, 'rb') as input_bam, \
         pysam.AlignmentFile(output_bam_filename, 'wb', template=input_bam) as output_bam:

        # Define the chunk size
        chunk_size = 10000

        # Define a lock to ensure thread safety when writing to the output bam file
        lock = threading.Lock()

        # Process the bam file in chunks
        chunk_iter = input_bam.fetch(until_eof=True)
        with concurrent.futures.ThreadPoolExecutor(max_workers=num_processes) as executor:
            # Process the reads in the chunk in parallel
            futures = [executor.submit(process_read, read) for read in chunk_iter]

            # Write the reads that passed the filter to the output bam file
            for future in concurrent.futures.as_completed(futures):
                read = future.result()
                if read is not None:
                    with lock:
                        output_bam.write(read)

In [3]:
extract_reads(input_bam_file, output_bam_file, num_threads)